# LoRAForge — Phase 2 on a free T4

Run this only after `notebooks/loraforge_t4.ipynb` produced `outputs/base-validation.json`
and `outputs/qlora-setup.json`, and after those files are back in the repository.

Order is part of the protocol: train, select on validation, freeze the selection, and only
then touch the publisher test split — exactly once. Do not reorder these cells.


In [ ]:
from pathlib import Path

# Works on Colab (/content) and Kaggle (/kaggle/working).
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'loraforge-llm'
if not (REPO / '.git').exists():
    !git clone --depth 1 https://github.com/mghadia1/loraforge-llm.git {REPO}
assert (REPO / '.git').exists(), 'clone failed; inspect the git output above'
%cd {REPO}
!python -m pip install -q -e ".[gpu]"


In [ ]:
import sys, site
site.main()  # an editable install adds a .pth the running kernel has not read
sys.path.insert(0, str(REPO / 'src'))
import loraforge
print('loraforge importable from', loraforge.__file__)


In [ ]:
import json, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name, f'Frozen evidence run expects a T4, found {gpu_name}'
assert Path('outputs/base-validation.json').exists(), 'run phase 1 and restore its artifact first'
assert Path('outputs/qlora-setup.json').exists(), 'run phase 1 and restore its artifact first'
print({'gpu': gpu_name, 'torch': torch.__version__})


In [ ]:
!python -m pytest -q


## Step 4 — two frozen epochs

`train_qlora` re-scores the untuned base with the adapter disabled first and refuses to
continue unless it reproduces the phase-one baseline macro-F1. It then trains the frozen
two epochs, saves and scores a checkpoint after each one, and selects the higher validation
macro-F1 (an exact tie goes to the earlier epoch). Loss sees only the answer code and EOS.


In [ ]:
from loraforge.config import default_config
from loraforge.data import load_dataset
from loraforge.modeling import load_quantized_base
from loraforge.qlora import attach_lora
from loraforge.training import train_qlora

config = default_config()
bundle = load_dataset(allow_test=False, config=config.data)
assert bundle.test is None

base_model, tokenizer = load_quantized_base(config)
model = attach_lora(base_model, config)
report = train_qlora(model, tokenizer, bundle, config, root=Path('.'))
print(json.dumps({
    'selected_epoch': report['selection']['selected_epoch'],
    'epoch_macro_f1': {entry['epoch']: entry['validation']['macro_f1'] for entry in report['epochs']},
    'base_validation_macro_f1': report['base_validation_metrics']['macro_f1'],
    'wall_time_seconds': report['wall_time_seconds'],
    'peak_cuda_memory_gib': report['peak_cuda_memory_gib'],
}, indent=2))


## Step 5a — freeze the selection before test exists

This is GPU-free. It recomputes every validation number from the saved logits, re-derives the
winning epoch from the rule, checks the adapter hashes, and fits a **separate** temperature for
the base and tuned systems on validation only. It refuses to run twice.


In [ ]:
!python -m loraforge.cli freeze-selection --root .
!python -m loraforge.cli verify --root .


## Restart the runtime here

Free the training model before the final evaluation: **Runtime → Restart runtime**, then run
the two cells below and nothing else.


## Step 5b — the single publisher-test evaluation

This loads all 7,600 held-out rows once and scores them with the same prompt twice: adapter
disabled (base) and adapter enabled (tuned). It refuses to run without the exact confirmation
string, refuses if a final report already exists, and refuses if the adapter changed after
freezing. Whatever the delta is — including a negative one — it gets written down.


In [ ]:
from pathlib import Path
import json

# The runtime was restarted, so redefine the paths before importing anything.
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'loraforge-llm'
%cd {REPO}
import sys
sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
from loraforge.config import default_config
from loraforge.final_test import CONFIRMATION, run_final_test, verify_final_report

report = run_final_test(default_config(), confirmation=CONFIRMATION, root=Path('.'))
base = report['systems']['base']
tuned = report['systems']['tuned']
print(json.dumps({
    'base_macro_f1': base['metrics_before_temperature']['macro_f1'],
    'tuned_macro_f1': tuned['metrics_before_temperature']['macro_f1'],
    'macro_f1_delta': report['delta']['macro_f1'],
    'base_ece_before': base['metrics_before_temperature']['calibration']['ece'],
    'base_ece_after': base['metrics_after_temperature']['calibration']['ece'],
    'tuned_ece_before': tuned['metrics_before_temperature']['calibration']['ece'],
    'tuned_ece_after': tuned['metrics_after_temperature']['calibration']['ece'],
}, indent=2))


In [ ]:
print(json.dumps(verify_final_report(root=Path('.')), indent=2))


## Download before closing the runtime

Bring back `outputs/` (reports and logits) and `adapters/selected/`. Locally, `loraforge verify`
must reproduce every number from those files before anything is written into the README, and
before Mayank's oral explanation gate is even attempted.
